# Simple RAG — RAG-EDU-1 (IZOHLANGAN VERSIYA)

Educational document-based question-answering system with answer generation (RAG).

Bu notebook har bir qatorda **nima uchun** shu kod yozilgani tushuntirilgan holda tayyorlangan — o'quv maqsadida.

Run the cells **in order, from top to bottom**.

## Step 1 — Install libraries and configure API access

In [ ]:
# "!" belgisi Colab'ga bu Python kodi emas, terminal buyrug'i ekanini bildiradi
# pip install -- Python kutubxonalarini yuklab o'rnatadi
# openai -- OpenAI'ning embedding va chat modellariga ulanish uchun kutubxona
# chromadb -- vektor bazasi (ma'lumotlarni saqlash uchun)
# -q ("quiet") -- o'rnatish jarayonining uzun log'ini yashiradi
!pip install -q openai chromadb

In [ ]:
import os                    # operatsion tizim bilan ishlash uchun (fayl/papka, muhit o'zgaruvchilari)
from getpass import getpass  # parol/kalit kiritganda ekranga ko'rinmaydigan qilib yashiradi

# os.environ -- bu "muhit o'zgaruvchilari" degan joy, dastur ichida maxfiy ma'lumotlarni saqlash uchun
# if not os.environ.get(...) -- "agar bu kalit hali o'rnatilmagan bo'lsa" (takror so'ramaslik uchun tekshiruv)
if not os.environ.get("OPENAI_API_KEY"):
    # getpass(...) -- sizdan API key so'raydi, lekin ekranda ko'rinmaydi (xavfsizlik uchun)
    # Kalit OPENAI_API_KEY nomli muhit o'zgaruvchisiga saqlanadi
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI  # OpenAI klassini import qilamiz -- so'rov yuborish uchun asosiy vosita

# client = OpenAI() -- "mijoz" obyekti yaratadi; u avtomatik OPENAI_API_KEY dan kalitni topib oladi
client = OpenAI()

# print() -- kod muvaffaqiyatli ishlaganini EKRANGA CHIQARISH uchun.
# Diqqat: agar print() bo'lmasa, ekranda hech narsa ko'rinmaydi, garchi kod to'g'ri ishlagan bo'lsa ham.
# Bu -- vizual tasdiq va debugging (xato topish) uchun foydali: agar bu xabar chiqmasa,
# demak yuqoridagi qatorlardan birida xatolik bo'lgan.
print("API client configured.")

## Step 2 — Prepare source documents

Choose ONE of the two options below.

In [ ]:
# --- OPTION A: Built-in demonstration documents ---

# demo_documents -- bu DICTIONARY (lug'at): {"fayl_nomi": "matn"} juftliklari.
# Har bir kalit (key) -- fayl nomi, qiymati (value) -- o'sha faylning matni.
demo_documents = {
    "company_policy.txt": (
        "Remote Work Policy\n"
        "Employees may work remotely up to 3 days per week with manager approval. "
        "Full remote work requires HR director sign-off and is reviewed quarterly. "
        "All remote employees must be reachable during core hours, 10:00 to 15:00 local time."
    ),
    "vacation_policy.txt": (
        "Vacation Policy\n"
        "Full-time employees accrue 20 paid vacation days per year. "
        "Unused days may be carried over to the next year up to a maximum of 5 days. "
        "Vacation requests must be submitted at least 2 weeks in advance."
    ),
    "onboarding_guide.md": (
        "# Onboarding Guide\n"
        "New employees complete a 5-day onboarding program. "
        "Day 1 covers company culture and IT setup. "
        "Day 2-3 cover role-specific training with a mentor. "
        "Day 4-5 include shadowing a senior team member and a final check-in with HR."
    ),
}

# os.makedirs(...) -- "source_docs" nomli papka yaratadi
# exist_ok=True -- agar papka allaqachon mavjud bo'lsa, xato bermaydi (davom etaveradi)
os.makedirs("source_docs", exist_ok=True)

# demo_documents.items() -- lug'atdagi har bir (fayl nomi, matn) juftligini birma-bir aylanib chiqadi
for filename, content in demo_documents.items():
    # os.path.join("source_docs", filename) -- papka nomi va fayl nomini TO'G'RI FORMATDA birlashtiradi.
    # Nega shunchaki "+" bilan qo'shib bo'lmaydi: Linux/Mac "/" ishlatadi, Windows esa "\" ishlatadi.
    # os.path.join() qaysi operatsion tizimda ishlayotganini avtomatik biladi va to'g'ri belgi qo'yadi.
    # Natija: "source_docs/company_policy.txt" kabi to'liq manzil (path) hosil bo'ladi.
    with open(os.path.join("source_docs", filename), "w", encoding="utf-8") as f:
        # open(..., "w", ...) -- faylni YOZISH rejimida ochadi ("w" = write)
        # encoding="utf-8" -- turli tillardagi harflar to'g'ri saqlanishi uchun
        # with ... as f: -- faylni ochadi va ishi tugagach AVTOMATIK yopadi (xavfsiz usul)
        f.write(content)  # matnni faylga yozadi

print(f"{len(demo_documents)} demonstration documents written to ./source_docs")

In [ ]:
# --- OPTION B: Upload your own .txt / .md documents ---

# google.colab.files -- faqat Colab muhitida ishlaydigan maxsus modul,
# kompyuterdan fayl yuklash oynasini ochadi
from google.colab import files
import shutil  # fayllarni ko'chirish/nusxalash uchun kutubxona

os.makedirs("source_docs", exist_ok=True)  # "source_docs" papkasini yaratadi (agar hali yo'q bo'lsa)

# files.upload() -- FAYL TANLASH OYNASINI OCHADI ("Browse..." tugmasi shu yerdan chiqadi).
# Yuklangan fayllar "uploaded" nomli lug'atga saqlanadi:
# {"physics_mechanics.txt": <fayl mazmuni baytlarda>}
uploaded = files.upload()

# uploaded.keys() -- yuklangan fayllar nomlarini oladi.
# Fayl avval Colab'ning asosiy papkasiga tushadi (yuklashning standart joyi).
for filename in uploaded.keys():
    # shutil.move(...) -- faylni "source_docs" papkasiga KO'CHIRADI (tartib saqlash uchun,
    # chunki Step 3 aynan shu papkani o'qiydi)
    shutil.move(filename, os.path.join("source_docs", filename))

# f"..." -- bu F-STRING, matn ichida {} orasiga o'zgaruvchi qo'yish imkonini beradi
print(f"Uploaded files moved to ./source_docs: {list(uploaded.keys())}")

## Step 3 — Split documents into fragments (chunking)

In [ ]:
# Bu qatorlar -- barcha muhim sozlamalarni BITTA JOYDA to'plash uchun (TS Ilova A parametrlari).
# Ularni alohida o'zgaruvchi qilib yozish yaxshi amaliyot: keyin CHUNK_SIZE'ni o'zgartirmoqchi
# bo'lsangiz, faqat shu bir joyni tuzatasiz, butun kodni qidirib yurish shart emas.

CHUNK_SIZE = 500      # har bir fragment (bo'lak) taxminan 500 belgidan iborat bo'ladi (tavsiya: 300-800)
CHUNK_OVERLAP = 80    # qo'shni bo'laklar bir-biri bilan 80 belgi ustma-ust tushadi (tavsiya: 50-100)
TOP_K = 4             # savol berilganda bazadan eng yaqin 4 ta bo'lak olinadi (tavsiya: 3-5)
EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4o-mini"
TEMPERATURE = 0        # modelning "ijodkorlik" darajasi; 0 = eng faktik, tasodifiylik yo'q


def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Splits text into overlapping fragments of fixed size."""
    # """...""" -- bu DOCSTRING, funksiya nima qilishini tushuntiruvchi izoh
    # (kod ishlashiga ta'sir qilmaydi, faqat hujjatlashtirish uchun)

    chunks = []       # bo'sh ro'yxat -- natijalar shu yerga yig'iladi
    start = 0          # matnni bo'lishni 0-belgidan boshlaymiz
    text = text.strip()  # matn boshi/oxiridagi ortiqcha bo'sh joy/qator ko'chirishlarni tozalaydi

    while start < len(text):  # matn tugamaguncha davom etadigan tsikl
        end = start + chunk_size  # bo'lakning tugash nuqtasi (masalan start=0, chunk_size=500 -> end=500)

        # text[start:end] -- Python'da matnni KESIB OLISH (slicing): start'dan end'gacha bo'lgan qism
        chunk = text[start:end].strip()

        if chunk:  # agar bo'lak bo'sh bo'lmasa (masalan matn oxirida bo'sh joy qolib ketmasin)
            chunks.append(chunk)  # bo'lakni ro'yxatga qo'shadi

        # ENG MUHIM QATOR: keyingi bo'lak qayerdan boshlanishini belgilaydi.
        # Nega "overlap" (ustma-ust tushish) kerak?
        # Agar gap ikkita bo'lak orasida "kesilib" qolsa, muhim ma'lumot yo'qolib qolishi mumkin.
        # Overlap -- ikki bo'lak orasida ma'lum miqdorda matnni QAYTARISH, shunda muhim jumla
        # ikkala bo'lakda ham to'liq saqlanib qoladi.
        # Misol: chunk_size=500, overlap=80 bo'lsa:
        #   1-bo'lak: belgi 0-500
        #   2-bo'lak: belgi 420-920 (500-80=420 dan boshlanadi, oxirgi 80 belgi qaytarilgan)
        start += chunk_size - overlap

    return chunks


all_chunks = []        # barcha hujjatlardan olingan BARCHA bo'laklar shu yerga yig'iladi
all_chunk_sources = []  # har bir bo'lak QAYSI FAYLDAN olinganini alohida (parallel) ro'yxatda saqlaydi

source_dir = "source_docs"

# os.listdir(source_dir) -- "source_docs" papkasidagi barcha fayl nomlarini ro'yxat qilib beradi
# sorted(...) -- fayllarni alifbo tartibida saralaydi (natija har doim bir xil tartibda chiqishi uchun)
for filename in sorted(os.listdir(source_dir)):
    # agar fayl .txt yoki .md bilan tugamasa (masalan PDF bo'lsa), uni O'TKAZIB YUBORADI
    # (continue -- tsiklning qolgan qismini bajarmay, keyingi faylga o'tadi)
    if not filename.endswith((".txt", ".md")):
        continue

    # open(..., "r", ...) -- faylni O'QISH rejimida ochadi ("r" = read)
    with open(os.path.join(source_dir, filename), "r", encoding="utf-8") as f:
        text = f.read()  # butun fayl mazmunini bitta katta matn (string) sifatida oladi

    chunks = split_into_chunks(text)  # yuqoridagi funksiyani chaqiradi, matnni bo'laklarga bo'ladi

    # all_chunks.extend(chunks) -- yangi bo'laklarni umumiy ro'yxatga qo'shadi.
    # extend -- ro'yxat ICHIGA ro'yxat qo'shadi (append'dan farqli, append butun ro'yxatni
    # bitta element sifatida qo'shib qo'yardi)
    all_chunks.extend(chunks)

    # [filename] * len(chunks) -- masalan 12 ta bo'lak chiqqan bo'lsa, bu fayl nomini
    # 12 marta takrorlangan ro'yxat qiladi, shunda har bir bo'lakka mos fayl nomi to'g'ri keladi
    all_chunk_sources.extend([filename] * len(chunks))

# set(all_chunk_sources) -- takrorlanmas fayl nomlarini oladi (nechta noyob hujjat borligini sanash uchun)
print(f"Loaded {len(set(all_chunk_sources))} documents, split into {len(all_chunks)} fragments.")
print("\nExample fragment:\n---")
print(all_chunks[0])  # birinchi bo'lakni namuna sifatida ko'rsatadi (natija to'g'ri chiqqanini tekshirish uchun)

## Step 4 — Vectorize and store fragments

In [ ]:
import chromadb

# chromadb.Client() -- IN-MEMORY (RAM'dagi) vektor bazasi klienti yaratadi.
# Eslatma: bu ma'lumotlar diskka yozilmaydi, Colab sessiyasi tugasa yo'qoladi.
chroma_client = chromadb.Client()

# try / except -- XATONI QAYTA ISHLASH konstruktsiyasi: "harakat qilib ko'r, xato chiqsa e'tibor berma".
# Nega kerak: agar Step 4 IKKINCHI marta ishga tushirilsa (masalan yangi hujjat qo'shib),
# avvalgi "documents" nomli collection allaqachon mavjud bo'ladi -- uni o'chirib, yangisini
# yaratish kerak. Agar collection hali mavjud bo'lmasa (birinchi marta), delete_collection
# xato beradi -- shuni except orqali e'tiborsiz qoldiramiz.
try:
    chroma_client.delete_collection("documents")
except Exception:
    pass

# "documents" nomli YANGI, BO'SH collection yaratadi -- bu yerga vektorlar, matnlar,
# manba nomlari saqlanadi (collection -- ChromaDB'da jadval/papka kabi narsa)
collection = chroma_client.create_collection(name="documents")


def embed_texts(texts, model=EMBEDDING_MODEL):
    """Converts a list of texts into embedding vectors via the OpenAI API."""
    # client.embeddings.create(...) -- Step 1'da yaratgan client orqali OpenAI'ning
    # embedding API'siga so'rov yuboradi.
    # input=texts -- qaysi matnlarni vektorga aylantirish kerakligi
    # model=model -- qaysi embedding modeli ishlatiladi (text-embedding-3-small)
    #
    # EMBEDDING NIMA? -- har bir matnni ko'p o'lchamli raqamlar ro'yxatiga (vektorga) aylantirish,
    # masalan [0.023, -0.041, 0.089, ...] (1536 ta son). Bu raqamlar matnning MA'NOSINI ifodalaydi --
    # ma'no jihatdan yaqin matnlar, vektor fazosida ham bir-biriga yaqin joylashadi.
    response = client.embeddings.create(input=texts, model=model)

    # response.data -- API'dan qaytgan natija, har bir matn uchun bitta obyekt (.embedding maydoni bilan)
    # [item.embedding for item in response.data] -- LIST COMPREHENSION: har bir natijadan faqat
    # .embedding qismini olib, yangi ro'yxat yasaydi
    return [item.embedding for item in response.data]


# Step 3'da tayyorlangan BARCHA bo'laklarni BITTA so'rovda OpenAI'ga yuboradi.
# Bu bitta API chaqiruvi orqali amalga oshiriladi (har bir bo'lak uchun alohida so'rov emas) --
# bu tezroq va arzonroq.
embeddings = embed_texts(all_chunks)

# collection.add(...) -- ChromaDB'ga yangi yozuvlar qo'shadi.
# To'rtta parametr PARALLEL ravishda beriladi: birinchi id + birinchi embedding + birinchi
# document + birinchi metadata -- barchasi BIR XIL bo'lakka tegishli.
collection.add(
    # ids -- har bir yozuvga NOYOB identifikator: "chunk_0", "chunk_1", ...
    # ChromaDB har bir yozuvga id talab qiladi (bazaning "asosiy kaliti" kabi)
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],

    embeddings=embeddings,   # yuqorida hisoblangan vektorlar

    documents=all_chunks,    # bo'lakning ASL MATNI (keyinchalik javob shakllantirishda ishlatiladi)

    # metadatas -- har bir bo'lak uchun QO'SHIMCHA MA'LUMOT, bu holda faqat qaysi fayldan olingani.
    # Masalan: {"source": "physics_mechanics.txt"}
    metadatas=[{"source": src} for src in all_chunk_sources],
)

# collection.count() -- collection ichida nechta yozuv borligini qaytaradi.
# Bu son Step 3'da hisoblangan len(all_chunks) bilan bir xil bo'lishi kerak --
# agar farq qilsa, biror narsa noto'g'ri ishlagan degani.
print(f"Stored {collection.count()} fragments in the vector database.")

## Step 5 — Query function (retrieval + prompt + generation)

In [ ]:
# SYSTEM_INSTRUCTION -- modelga beriladigan "rol" ko'rsatmasi (system prompt), har bir so'rovda
# modelga qanday xulq-atvor kutilishini aytadi.
# Uch qatorli matn (...) ichida yozilgan -- Python'da qavs ichida yonma-yon yozilgan stringlar
# avtomatik BIRLASHADI (bitta uzun matn hosil bo'ladi, "+" yozish shart emas).
# Bu matn eng muhim talabni amalga oshiradi: model FAQAT berilgan kontekstdan javob bersin,
# o'zidan to'qib chiqarmasin (hallutsinatsiyaning oldini olish).
SYSTEM_INSTRUCTION = (
    "You are a helpful assistant that answers questions using ONLY the provided context. "
    "If the answer is not contained in the context, say clearly that the information is "
    "not available in the supplied documents. Do not invent or guess information. "
    "Keep answers concise and directly grounded in the context."
)


def ask(question, top_k=TOP_K, verbose=True):
    """
    Full RAG pipeline for a single question:
    1. Vectorize the question (same embedding model as indexing)
    2. Retrieve the nearest fragments from the vector database
    3. Assemble a prompt with context + question
    4. Call the generation model
    5. Return the answer with its sources
    """
    # question -- foydalanuvchi savoli
    # top_k=TOP_K -- nechta bo'lak olib kelinishi (standart holatda Step 3'da belgilangan 4)
    # verbose=True -- agar True bo'lsa, natijalarni ekranga chiroyli chiqaradi;
    #                  False qilsangiz, faqat natijani qaytaradi (Step 6 testlarida foydali)

    # --- 1-2. Savolni vektorlashtirish va qidirish ---

    # embed_texts([question]) -- Step 4'dagi funksiyani chaqiradi, savolni BITTA ELEMENTLI
    # ro'yxat qilib beradi (chunki funksiya ro'yxat kutadi).
    # [0] -- natija ro'yxatidan BIRINCHI (yagona) vektorni oladi.
    # MUHIM: savol ham xuddi hujjat bo'laklari kabi, AYNAN SHU embedding modeli bilan
    # vektorlashtiriladi -- indeksatsiya va so'rov bir xil modeldan foydalanishi shart,
    # aks holda vektorlar taqqoslanmaydi.
    query_embedding = embed_texts([question])[0]

    # collection.query(...) -- ChromaDB'ga: "menga shu vektorga eng yaqin top_k ta yozuvni top" so'rovi
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)

    # results -- ChromaDB'dan qaytgan javob, lug'at ko'rinishida ("documents", "metadatas" kalitlari bilan)
    # results["documents"] -- topilgan bo'laklarning ASL MATNI, lekin ro'yxat ichida ro'yxat
    # ([[bo'lak1, bo'lak2, ...]]), chunki bir vaqtda bir nechta savol yuborilishi mumkin edi.
    # Bizda faqat bitta savol bo'lgani uchun [0] bilan tashqi ro'yxatning birinchi elementini olamiz.
    retrieved_chunks = results["documents"][0]

    # [meta["source"] for meta in results["metadatas"][0]] -- list comprehension: har bir
    # metadata'dan faqat "source" qiymatini ajratib, yangi ro'yxat yasaydi
    retrieved_sources = [meta["source"] for meta in results["metadatas"][0]]

    # HIMOYA QATORI (edge case handling): agar hech qanday bo'lak topilmasa
    # (masalan Step 4 ishga tushirilmagan bo'lsa), funksiya darhol to'xtaydi
    if not retrieved_chunks:
        return {"answer": "No documents are indexed yet.", "sources": []}

    # --- 3. Prompt yig'ish ---

    # zip(retrieved_chunks, retrieved_sources) -- ikkita ro'yxatni JUFTLASHTIRIB birlashtiradi:
    # birinchi bo'lak + birinchi manba, ikkinchi bo'lak + ikkinchi manba, va h.k.
    # f"[Source: {src}]\n{chunk}" -- har bir bo'lakni formatlaydi: avval qaysi fayldan ekanini
    # ko'rsatadi, keyin yangi qatorda bo'lak matnini qo'yadi.
    # "\n\n---\n\n".join(...) -- barcha formatlangan bo'laklarni ikki bo'sh qator va chiziqcha
    # bilan ajratib, bitta katta matn qilib birlashtiradi (vizual ravishda ajratish uchun)
    context_block = "\n\n---\n\n".join(
        f"[Source: {src}]\n{chunk}"
        for chunk, src in zip(retrieved_chunks, retrieved_sources)
    )

    # Yakuniy prompt uch qismdan iborat: KONTEKST (topilgan bo'laklar) + SAVOL +
    # qat'iy ko'rsatma ("faqat yuqoridagi kontekstdan javob ber").
    # Bu -- RAG'ning yuragi: model o'z xotirasidan emas, aynan shu berilgan matndan
    # javob berishga majburlanadi.
    user_prompt = (
        f"Context:\n{context_block}\n\n"
        f"Question: {question}\n\n"
        "Answer using only the context above."
    )

    # --- 4. Modelni chaqirish ---

    # client.chat.completions.create(...) -- OpenAI'ning chat modeliga (gpt-4o-mini) so'rov yuboradi.
    # messages -- suhbat tarixi, ikkita "rol" bilan:
    #   "system" -- modelga umumiy ko'rsatma (SYSTEM_INSTRUCTION)
    #   "user"   -- foydalanuvchi xabari (kontekst + savol)
    # temperature=TEMPERATURE -- 0 qiymati, model eng ehtimoliy, qat'iy javobni beradi
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": user_prompt},
        ],
    )

    # response.choices[0] -- model bir nechta variant javob taklif qilishi mumkin,
    # biz BIRINCHISINI olamiz (odatda faqat bitta bo'ladi).
    # .message.content -- javobning matn qismini oladi
    answer = response.choices[0].message.content

    # --- 5. Natijani qaytarish ---

    # set(retrieved_sources) -- takrorlanmas fayl nomlarini oladi (bir xil fayldan bir nechta
    # bo'lak bo'lsa, bir marta ko'rsatish uchun)
    # sorted(...) -- alifbo tartibida saralaydi
    unique_sources = sorted(set(retrieved_sources))

    if verbose:
        print("Answer:\n", answer)
        print("\nSources:", ", ".join(unique_sources))
        print("\nRetrieved fragments (for verification):")
        # chunk[:100] -- har bir topilgan bo'lakning BIRINCHI 100 belgisi -- bu sizga
        # retrieval to'g'ri ishlaganini tekshirish imkonini beradi (masalan, savolga
        # aloqasi yo'q bo'lak topilgan bo'lsa, shu yerda ko'rinadi)
        for chunk, src in zip(retrieved_chunks, retrieved_sources):
            print(f"  [{src}] {chunk[:100]}...")

    # Funksiya oxirida LUG'AT qaytaradi -- bu Step 6'dagi testlarda result["answer"],
    # result["sources"] kabi qulay foydalanish uchun
    return {"answer": answer, "sources": unique_sources, "retrieved_chunks": retrieved_chunks}

In [ ]:
# Sinab ko'ramiz (demo hujjatlar bilan; agar physics_mechanics.txt yuklagan bo'lsangiz,
# savolni fizikaga oid qilib o'zgartiring, masalan: "What is Newton's second law?")
_ = ask("How many vacation days do employees get per year?")

## Step 6 — Functional verification and acceptance testing

In [ ]:
# Bu Step -- TS 6-bo'limidagi 5 ta acceptance mezonini QO'LDA (ko'z bilan) tekshirish jarayoni.
# Umumiy qolip har bir test uchun bir xil:
#   1) sarlavha chop etiladi
#   2) ask(..., verbose=False) chaqiriladi -- verbose=False, chunki natijani o'zimiz
#      qanday ko'rsatishni nazorat qilamiz (Step 5'dagi ichki print ishlamaydi)
#   3) result -- lug'at ({"answer":..., "sources":..., "retrieved_chunks":...})
#   4) result["answer"], result["sources"] -- lug'atdan kerakli qismlarni kalit orqali olamiz

print("=== Test 1: Correctness of fragment retrieval (known answer) ===")
# Ma'lum javobi bor savol -- to'g'ri bo'lak topilganini ko'z bilan tekshiramiz
result = ask("How many days can employees carry over from unused vacation?", verbose=False)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print()

print("=== Test 2: Answer reliability (matches document content) ===")
# Javob hujjat matniga mos kelishi, o'zidan hech narsa qo'shmasligi kerak
result = ask("What are the core hours for remote employees?", verbose=False)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print()

print("=== Test 3: Behaviour when data is absent ===")
# ENG MUHIM TEST: bu savol hujjatda UMUMAN YO'Q mavzu haqida.
# Agar model "ma'lumot yo'q" demasdan o'zidan nimadir to'qib chiqarsa,
# demak SYSTEM_INSTRUCTION yetarli kuchli emas, tuzatish kerak bo'ladi.
result = ask("What is the company's policy on parental leave?", verbose=False)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("(Expected: the system should state this information is not available.)")
print()

print("=== Test 4: Source attribution ===")
# result["sources"] bo'sh emasligini va to'g'ri fayl nomini ko'rsatayotganini tekshiramiz
result = ask("What happens on day 1 of onboarding?", verbose=False)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("(Expected: sources list is non-empty and correct.)")
print()

print("=== Test 5: Reproducibility (re-run with unchanged data) ===")
# Bir xil savol IKKI MARTA so'raladi. TEMPERATURE=0 bo'lgani uchun (Step 3),
# model har safar deyarli bir xil javob beradi -- tasodifiylik yo'q.
# Agar ikkala natija bir xil bo'lsa -- tizim BARQAROR (reproducible).
result_a = ask("How many vacation days do employees get per year?", verbose=False)
result_b = ask("How many vacation days do employees get per year?", verbose=False)
print("Run A sources:", result_a["sources"])
print("Run B sources:", result_b["sources"])
print("(Expected: same sources retrieved both times, since temperature=0 and data is unchanged.)")

## Using your own documents

1. Step 2'da Option B (upload) cell'ni ishga tushiring.
2. Step 3 (chunking) va Step 4 (vectorize/store) ni QAYTA ishga tushiring, shunda yangi hujjatlar indekslanadi.
3. `ask("savolingiz")` funksiyasidan foydalaning.

**Muhim:** hujjatlarni almashtirgach har doim Step 3-4'ni qayta ishga tushiring — vektor baza faqat oxirgi marta Step 4 ishga tushirilganda nima indekslangan bo'lsa, o'shani aks ettiradi.

## Ask your own question

In [ ]:
# Savolni o'zgartirib, shu cell'ni ishga tushiring:
_ = ask("Your question here")